In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

In [3]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

clean_path = PROCESSED_DIR / "cleaned_churn_data.csv"

df = pd.read_csv(clean_path)

df.shape

(7043, 21)

In [4]:
df["tenure_group"] = pd.cut(
    df["tenure"],
    bins=[0, 12, 24, 48, 60, 72],
    labels=["0-12", "12-24", "24-48", "48-60", "60-72"],
    right=False
)

In [5]:
df["tenure_group"].value_counts()

tenure_group
0-12     2069
24-48    1624
60-72    1121
12-24    1047
48-60     820
Name: count, dtype: int64

In [6]:
service_cols = [
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df["num_services"] = df[service_cols].apply(lambda x: (x == "Yes").sum(), axis=1)

In [7]:
df["num_services"].describe()

count    7043.000000
mean        2.459747
std         2.045539
min         0.000000
25%         1.000000
50%         2.000000
75%         4.000000
max         7.000000
Name: num_services, dtype: float64

In [8]:
df["monthly_charge_group"] = pd.cut(
    df["MonthlyCharges"],
    bins=[0, 35, 70, 100, 150],
    labels=["Low", "Medium", "High", "Very High"]
)

In [9]:
df["monthly_charge_group"].value_counts()

monthly_charge_group
High         2681
Low          1735
Medium       1725
Very High     902
Name: count, dtype: int64

In [10]:
df["customer_value_segment"] = pd.qcut(
    df["TotalCharges"],
    q=4,
    labels=["Low Value", "Mid Value", "High Value", "Very High Value"]
)

In [11]:
df["customer_value_segment"].value_counts()

customer_value_segment
Low Value          1762
Very High Value    1761
Mid Value          1760
High Value         1760
Name: count, dtype: int64

Since the dataset lacked a direct lifetime value metric, I segmented customers using TotalCharges to identify high-value customers and analyze churn risk by revenue contribution.

In [12]:
df["is_month_to_month"] = (df["Contract"] == "Month-to-month").astype(int)

In [13]:
df["is_new_customer"] = (df["tenure"] < 12).astype(int)

In [14]:
df["paperless_billing_flag"] = df["PaperlessBilling"]

In [15]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,MonthlyCharges,TotalCharges,Churn,tenure_group,num_services,monthly_charge_group,customer_value_segment,is_month_to_month,is_new_customer,paperless_billing_flag
0,7590-VHVEG,Female,0,1,0,1,0,No,DSL,No,...,29.85,29.85,0,0-12,1,Low,Low Value,1,1,1
1,5575-GNVDE,Male,0,0,0,34,1,No,DSL,Yes,...,56.95,1889.50,0,24-48,2,Medium,High Value,0,0,0
2,3668-QPYBK,Male,0,0,0,2,1,No,DSL,Yes,...,53.85,108.15,1,0-12,2,Medium,Low Value,1,1,1
3,7795-CFOCW,Male,0,0,0,45,0,No,DSL,Yes,...,42.30,1840.75,0,24-48,3,Medium,High Value,0,0,0
4,9237-HQITU,Female,0,0,0,2,1,No,Fiber optic,No,...,70.70,151.65,1,0-12,0,High,Low Value,1,1,1


In [16]:
analytics_path = PROCESSED_DIR / "analytics_ready_churn_data.csv"
df.to_csv(analytics_path, index=False)

analytics_path

WindowsPath('C:/Users/shrut/OneDrive/Desktop/customer-churn-platform/data/processed/analytics_ready_churn_data.csv')

Feature Engineering Summary:

Added Features:

1. tenure_group - Grouped customers by tenure to identify lifecycle churn patterns.
2. num_services - Count of services used. Higher service usage indicates stronger engagement and lower churn risk.
3. monthly_charge_group - Categorized monthly charges to analyze price sensitivity and churn behavior.
4. customer_value_segment - Segmented customers by total charges (proxy for lifetime value) to identify high-value customers and revenue impact.
5. is_month_to_month - Flags month-to-month contracts, which are associated with higher churn risk.
6. is_new_customer - Identifies customers with tenure under 12 months, a high-risk churn period.
7. paperless_billing_flag - Retained for behavioral analysis and potential churn correlation.